# Stage 18 — lexicon CEILING diagnostic (val only, free)
Stage 17b gave +0.0047 at 49% coverage. Is that a coverage problem or is the
model output too noisy to snap? This runs lexicon decoding on **val** at:
  * TRAIN-lex lexicon (~50% coverage, leakage-free) -- the realistic baseline
  * VAL-vocab lexicon (100% coverage, LEAKAGE -- diagnostic CEILING only)
with improved edit-distance candidates + CTC rescoring. If the 100% ceiling
barely beats greedy, lexicon decoding is dead regardless of word list -> go to
item 3 (augmentation). If it's a big drop, item 2 is the headline lever and we
find the real external 6000-word list. **No test, no marker.**

Attach the landmark cache + Stage 11 checkpoint.


## Cell 1 — clone + deps


In [ ]:
import sys, subprocess as sp
sp.run('rm -rf /kaggle/working/wita_v2', shell=True)
sp.run("git clone -b stage13b-paper-replication "
       "'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'", shell=True, check=True)
sys.path.insert(0, '/kaggle/working/wita_v2')
for _m in [m for m in sys.modules if m.split('.')[0] in ('stage16','stage17','stage18','datasets','models')]:
    del sys.modules[_m]
sp.run('pip -q install editdistance', shell=True)
print('ready')


## Cell 2 — load model + cache


In [ ]:
import os, glob, torch
from stage17.common import find_landmark_cache
from stage17.seq_lexicon_decode import (CTCConverter, load_stage11, collect_lex_words,
                                        build_flat_lexicon, evaluate_split_flat)
LM_CACHE = find_landmark_cache(preferred=['/kaggle/input/datasets/gaurs86/wita-full-english-landmark-cache'])
cands = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True)+glob.glob('/kaggle/input/**/*.pt', recursive=True))
CKPT = ([c for c in cands if 'stage11' in c.lower() or 'best' in c.lower()] or cands)[0]
print('cache', LM_CACHE, '| ckpt', CKPT)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
conv = CTCConverter(); model = load_stage11(CKPT, device)


## Cell 3 — ceiling: train-lex lexicon vs val-vocab (100%) lexicon, on VAL


In [ ]:
from stage17.seq_lexicon_decode import print_result
train_words = collect_lex_words(LM_CACHE, 'train')
val_words   = collect_lex_words(LM_CACHE, 'val')
print(f'train-lex words={len(train_words)}  val-lex words={len(val_words)}')

print('\n=== A) TRAIN-lex lexicon (leakage-free, ~50% coverage) ===')
flatA = build_flat_lexicon(train_words, conv)
rA = evaluate_split_flat(model, LM_CACHE, 'val', conv, flatA, train_words, device, max_ed=2, topk=40)
print_result(rA)

print('\n=== B) VAL-vocab lexicon (100% coverage, LEAKAGE -- CEILING ONLY) ===')
flatB = build_flat_lexicon(val_words, conv)
rB = evaluate_split_flat(model, LM_CACHE, 'val', conv, flatB, val_words, device, max_ed=2, topk=40)
print_result(rB)

print('\n================ VERDICT ================')
print(f"greedy lex                 = {rA['greedy']['lex']:.4f}")
print(f"+lexicon (train, ~50% cov) = {rA['lexicon']['lex']:.4f}")
print(f"+lexicon (val,  100% CEIL) = {rB['lexicon']['lex']:.4f}   <-- ceiling")
print('If the 100% ceiling barely moves vs greedy -> lexicon decoding is dead (model too noisy).')
print('If it drops a lot -> coverage was the problem; item 2 is the lever, find the real 6000 list.')
